# Week 7 - Webscraping with a little bit of API's
- recap of api's
- requests
- BeautifulSoup
- combining scraping and API's

## Recap API's

In [3]:
#https://techy-api.vercel.app/
import json, requests

url = "https://techy-api.vercel.app/api/json"
response = requests.get(url)
text = response.text
data = json.loads(text)

print("Quote of the day:\n", data["message"])

Quote of the day:
 Make sure to modulate the fragmentation distortion


In [11]:
#finding whichever amount of films and their names a specific char plays is.
import json, requests

url = "https://api.disneyapi.dev/character"
response = requests.get(url)
data = json.loads(response.text)

#find baloo the bear and of course in the movies it plays
for chars in data["data"]:
    if chars["name"] == "Baloo":
        list = chars["films"]


media = []
for chars in data["data"]:
    if chars["name"] == "Baloo":
        media.append(chars["films"])
        media.append(chars["shortFilms"])
        media.append(chars["tvShows"])

print("Baloo starred in the following tv MEDIA:")
for m in media:
    for item in m:
        print(item)
    

print("\n")
print("Baloo starred in the following films:\n")
for film in list:
    print(film)


Baloo starred in the following tv MEDIA:
The Jungle Book
The Jungle Book 2
Mickey's Magical Christmas: Snowed in at the House of Mouse
Mickey's House of Villains
Once Upon a Studio
The Mouse Factory
Jungle Cubs
House of Mouse


Baloo starred in the following films:

The Jungle Book
The Jungle Book 2
Mickey's Magical Christmas: Snowed in at the House of Mouse
Mickey's House of Villains


In [ ]:
#find the char with the most films and also how many!
import json, requests

url = "https://api.disneyapi.dev/character"
response = requests.get(url)
data = json.loads(response.text)


counter = 0
for char in data["data"]:
    if len(char["films"]) > counter:
        counter = len(char["films"])
        name = char["name"]

print(f"char with the most films is {name}!")



#provide a list of characters with at least 1 videogame and 1 film
counter = 0
list = []
for char in data["data"]:
    if char["films"] and char["videoGames"]:
        list.append(char["name"])

#for the if statement it's also possible to do 
#--> len(char["films"]) OR char["films"] != [] OR char["videoGames"] is not null
print(list)


char with the most films is Baloo!
['Achilles', 'Baloo', 'Beheaded Knight', 'Captain Amelia']


## Scraping starts here!

In [ ]:
import requests
import re

url = "https://en.wikipedia.org/wiki/Film"
headers = {'User-Agent': 'Anthony'}
response = requests.get(url, headers=headers)
if response.status_code == 200:
    html = response.text
    regex = "film"
    matches = re.findall(regex, html.lower())
    print(len(matches))
else:
    print("try again, probably no access")

#be careful, triple the amount of results compared to a regular ctrl f on the page
#this includes class names etc in the html code


1646


## User BeautifulSoup to clean the htmlSoup

In [8]:
! python -m pip install beautifulsoup4


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Film"
headers = {'User-Agent': 'Anthony'}
response = requests.get(url, headers=headers)
html = response.text

soup = BeautifulSoup(html)
#print(soup.prettify())
print(soup.title)
print(soup.title.get_text())
print(soup.get_text().lower().count("film"))
print(soup.get_text().count("Film"))

print(soup.get_text()[0:500].split())

h3tags = soup.find_all("h3")
for tag in h3tags:
    print(tag.get_text())


<title>Film - Wikipedia</title>
Film - Wikipedia
515
113
['Film', '-', 'Wikipedia', 'Jump', 'to', 'content', 'Main', 'menu', 'Main', 'menu', 'move', 'to', 'sidebar', 'hide', 'Navigation', 'Main', 'pageContentsCurrent', 'eventsRandom', 'articleAbout', 'WikipediaContact', 'us', 'Contribute', 'HelpLearn', 'to', 'editCommunity', 'portalRecent', 'changesUpload', 'fileSpecial', 'pages', 'Search', 'Search', 'Appearance', 'Donate', 'Create', 'account', 'Log', 'in', 'Personal', 'tools', 'Donate', 'Create', 'account', 'Log', 'in']
Precursors
1830s–1880s: Before celluloid
1880s–1890s: First motion pictures
1910s: Early evolution
1920s–1960s: Evolution in sound
1930s: Evolution in color
1950s: growing influence of television
1960s–present: Modern cinema
Language
Montage
Film criticism
Preview
Trailer and teaser
Education and propaganda
Crew
Technology
Independent
Open content film
Fan film


In [ ]:
#getting all of the elephants in different languages
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Elephant"
headers = {'User-Agent': 'Anthony'}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text)

liTags = soup.find_all("li", {'class': "interlanguage-link"})
language = input()

for tag in liTags:
    if language in tag.a["title"]:
        print(tag.a["title"])

Слон – Bulgarian


In [54]:
#downloading images through our lovely soup --> not the easiest thing in the world to try and find the correct ones
import requests, re
from bs4 import BeautifulSoup
import time

url = "https://en.wikipedia.org/wiki/Film"
headers = {'User-Agent': 'Anthony'}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text)

imgTags = soup.find_all("img")
sources = [img["src"] for img in imgTags]
updatedSources = []
for source in sources:
    if ".jpg" in source or ".png" in source or ".gif" in source: 
        updatedSources.append(source)

for img in updatedSources:
    url = "https:" + img
    
    r = requests.get(url, headers=headers, allow_redirects=True)
    time.sleep(5)
    with open(r"Images\\" + img.split("/")[-1], 'wb') as file:
        file.write(r.content)
